In [1]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
import numpy as np
import geopandas as gpd

In [2]:
df = pd.read_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/all_results.csv')

In [3]:
print(df.columns.tolist())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [4]:
# unify the symbols output format
def clean_column_pattern(value, row_idx, col_name):
    if pd.isna(value):
        return value
    value_str = str(value)

    match = re.search(r'([a-zA-Z]+\d+)\)?', value_str)
    if match:
        return match.group(1)
    print(f"Row {row_idx}, Column '{col_name}': Unmatched value '{value_str}'")
    return value

columns_to_clean = [col for col in df.columns if '_Symbol' in col and col != 'Demo_which_state_of_the_usa_do_they_live_in__Symbol']
for col in columns_to_clean:
    df[col] = pd.Series(
        (clean_column_pattern(val, row_idx=idx, col_name = col) for idx, val in df[col].items()),
        index = df.index
    )

In [5]:
# Replace 'dia' and 'dib' prefixes with 'dic' in the column
df['Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol'] = df['Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol'].apply(
    lambda x: re.sub(r'^di[ab](\d+)$', r'dic\1', str(x)) if pd.notna(x) else x
)

In [6]:
# Standardize letter prefixes in symbol columns (sometimes there are a multiple letter prefix in one column)
for col in columns_to_clean:
    non_null_values = df[col].dropna()

    if len(non_null_values) == 0:
        print(f"Column '{col}' has no non-null values.\n")
        continue

    letter_prefixes = set()
    prefix_counts = {}
    for idx, val in non_null_values.items():
        match = re.match(r'([a-zA-Z]+)', str(val))
        if match:
            prefix = match.group(1)
            letter_prefixes.add(prefix)
            prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

    if len(letter_prefixes) > 1:
        print(f"Column '{col}' has multiple letter prefixes: {letter_prefixes}")
        for prefix, count in sorted(prefix_counts.items()):
            print(f"  {prefix}: {count}")
        # Find the most common prefix
        most_common_prefix = max(prefix_counts, key=prefix_counts.get)

        df[col] = df[col].apply(
            lambda x: re.sub(r'^[a-zA-Z]+', most_common_prefix, str(x)) if pd.notna(x) else x
        )
        print()

Column 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol' has multiple letter prefixes: {'imd', 'ima'}
  ima: 10490
  imd: 2

Column 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_the_usa_Symbol' has multiple letter prefixes: {'imc', 'ima', 'imb'}
  ima: 1
  imb: 3
  imc: 10488

Column 'Belief_opinion_on_what_should_be_done_with_illegal_immigrants_Symbol' has multiple letter prefixes: {'imd', 'imc'}
  imc: 2
  imd: 10490

Column 'Belief_opinion_on_the_statement_men_are_better_suited_emotionally_for_politics_than_are_most_women_Symbol' has multiple letter prefixes: {'feb', 'fea'}
  fea: 10486
  feb: 6

Column 'Belief_opinion_on_declining_share_of_white_people_in_america_Symbol' has multiple letter prefixes: {'raa', 'rab'}
  raa: 1
  rab: 10491

Column 'Belief_opinion_on_the_statement_because_of_past_discrimination_black_people_should_be_given_preference_in_hiring_and_promotion_Symbol' has multiple letter prefixes: {'fra', 'rac'}
  fra

In [7]:
df.head()

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Demo_interest_in_politics_Symbol,Demo_interest_in_politics_Speculation,Demo_marital_status_Category,Demo_marital_status_Explanation,Demo_marital_status_Symbol,Demo_marital_status_Speculation,Demo_partisan_loyalties_Category,Demo_partisan_loyalties_Explanation,Demo_partisan_loyalties_Symbol,Demo_partisan_loyalties_Speculation
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# df_state = df[['Demo_which_state_of_the_usa_do_they_live_in__Symbol']].copy()
# print(f"Number of rows with state category 'text)': {(df_state['Demo_which_state_of_the_usa_do_they_live_in__Symbol'] == 'text) ny').sum()}")
# # Find rows where state is 'text)'
text_rows = df[df['Demo_which_state_of_the_usa_do_they_live_in__Symbol'] == 'xx']
print(f"Example row with 'text)' state:\n")
print(text_rows[['Demo_which_state_of_the_usa_do_they_live_in__Symbol', 'Demo_which_state_of_the_usa_do_they_live_in__Explanation']].head(1))

# Print explaination for states with weird category lables
if len(text_rows) > 0:
    pd.set_option('display.max_colwidth', None)
    explanation = text_rows['Demo_which_state_of_the_usa_do_they_live_in__Explanation'].iloc[0]
    print("\nFull explanation:")
    print(explanation)

Example row with 'text)' state:

      Demo_which_state_of_the_usa_do_they_live_in__Symbol  \
14395                                                 xx    

      Demo_which_state_of_the_usa_do_they_live_in__Explanation  
14395  the profile states a selfreported location of ...        

Full explanation:
the profile states a selfreported location of paris, philadelphia. paris is in france and philadelphia is a city in the united states, but no u.s. state is mentioned. because the user clearly lists a nonu.s. city (paris) and only a u.s. city without a state, we conclude the user is not tied to any u.s. state. this is a highspeculation inference since the data give no explicit state information.


In [9]:
# Unified state cleanup mapping
STATE_MAPPING = {
    # Standard abbreviations
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'florida': 'FL', 'georgia': 'GA',
    'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA',
    'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS', 'missouri': 'MO',
    'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV', 'new hampshire': 'NH', 'new jersey': 'NJ',
    'new mexico': 'NM', 'new york': 'NY', 'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH',
    'oklahoma': 'OK', 'oregon': 'OR', 'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC',
    'south dakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT',
    'virginia': 'VA', 'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY',

    # D.C.
    'district of columbia': 'DC', 'washington dc': 'DC', 'dc': 'DC',

    # Comma variations
    'ny,': 'NY', 'wa,': 'WA',
    'california,': 'CA', 'new york,': 'NY', 'texas,': 'TX', 'florida,': 'FL',
    'illinois,': 'IL', 'massachusetts,': 'MA', 'michigan,': 'MI',
    'minnesota,': 'MN', 'arizona,': 'AZ', 'nevada,': 'NV', 'tennessee,': 'TN',
    'south carolina,': 'SC', 'kansas,': 'KS', 'louisiana,': 'LA', 'maine,': 'ME',
    'connecticut,': 'CT', 'utah,': 'UT', 'idaho,': 'ID',

    # Parenthetical variations
    'california (ca)': 'CA', 'arizona (az)': 'AZ', 'kansas (ks)': 'KS',
    'tennessee (tn)': 'TN', 'hi (hawaii)': 'HI', 'ma (massachusetts)': 'MA',
    'ca (california)': 'CA',

    # USA 
    'usa': 'USA', 'usa,': 'USA', 'us':'USA',
    'usa (no specific state)': 'USA',
    'usa (unspecified state)': 'USA',
    'usa (not a specific state)': 'USA',
    'usa (no specific state identified)': 'USA',
    'usa (the user is from the united states but does not specify a state)': 'USA',
    'usa (the user is from the united states but no specific state is identified)': 'USA',
    'if they do not specify a state, but are still from the united states of america, write "usa"': 'USA',

    # No state specified
    'text': 'No_state', 'na': 'No_state', 'n/a': 'No_state', 'xx': 'No_state',
    'not from a state in the usa': 'No_state',
    'not from a state in the usa,': 'No_state',
    'no state': 'No_state',
}

def clean_state(state):
    if pd.isna(state):
        return np.nan
    s = str(state).strip().lower().rstrip(')')
    # Remove 'text)' prefix if present
    if s.startswith('text)'):
        s = s[6:].strip()

    if s in STATE_MAPPING:
        return STATE_MAPPING[s]
    
    if len(s) == 2 and s.upper() in STATE_MAPPING.values():
        return s.upper()    
    
    # If direct mapping fails, try partial matches, longest keys first to avoid false positives
    for key in sorted(STATE_MAPPING.keys(), key=len, reverse=True):
        if key in s:
            return STATE_MAPPING[key]

    print(f"No match found: '{s}'")
    return s.upper()

df['state_cleaned'] = df['Demo_which_state_of_the_usa_do_they_live_in__Symbol'].apply(clean_state)
print(len(df['state_cleaned'].unique()))
print(df['state_cleaned'].value_counts())

51
state_cleaned
No_state    2582
CA           966
NY           761
TX           557
FL           551
USA          511
PA           350
WA           283
VA           254
AZ           219
MA           209
CO           188
GA           187
OH           181
TN           179
NJ           177
MN           169
IL           166
NC           165
MI           163
DC           154
NV           128
MD           127
OR           126
SC           122
WI            95
KS            93
UT            80
IA            80
IN            67
KY            65
LA            58
AL            54
CT            48
AR            42
HI            39
DE            36
MT            32
ID            32
NH            30
MO            25
WY            18
NM            17
MS            16
SD            15
OK            13
RI            11
VT            10
ME             9
NE             8
Name: count, dtype: int64


/tmp/ipykernel_10615/2858605418.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['state_cleaned'] = df['Demo_which_state_of_the_usa_do_they_live_in__Symbol'].apply(clean_state)


In [10]:
# add lickert order detection for belief columns
belief_cols = df.columns[df.columns.str.startswith('Belief_') & df.columns.str.endswith('_Symbol')]

def extract_number(cat):
    match = re.search(r"(\d+)$", str(cat))
    if match:
        return int(match.group(1))
    else:
        return None
    
def detect_likert_order(df, col):
    cats = df[col].dropna().unique()
    ordered_cats = sorted(cats, key=extract_number)
    return ordered_cats

for col in belief_cols:
    likert_order = detect_likert_order(df, col)

    df[col + '_cat'] = pd.Categorical(df[col], categories=likert_order, ordered=True)
    df[col + '_num'] = df[col + '_cat'].cat.codes + 1
    

/tmp/ipykernel_10615/1756759671.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_cat'] = pd.Categorical(df[col], categories=likert_order, ordered=True)
/tmp/ipykernel_10615/1756759671.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_num'] = df[col + '_cat'].cat.codes + 1
/tmp/ipykernel_10615/1756759671.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

In [11]:
df.to_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_all_result.csv')

In [12]:
df.to_pickle('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_all_result.pkl')   

# preprocess for stan model (drop USA and No_state, add predictor index)

In [3]:
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_all_result.pkl')    

In [4]:
df = df[(df['state_cleaned'] != 'No_state') & (df['state_cleaned'] != 'USA')]

In [5]:
print(len(df['state_cleaned'].unique()))
print(df['state_cleaned'].value_counts())

49
state_cleaned
CA    966
NY    761
TX    557
FL    551
PA    350
WA    283
VA    254
AZ    219
MA    209
CO    188
GA    187
OH    181
TN    179
NJ    177
MN    169
IL    166
NC    165
MI    163
DC    154
NV    128
MD    127
OR    126
SC    122
WI     95
KS     93
UT     80
IA     80
IN     67
KY     65
LA     58
AL     54
CT     48
AR     42
HI     39
DE     36
MT     32
ID     32
NH     30
MO     25
WY     18
NM     17
MS     16
SD     15
OK     13
RI     11
VT     10
ME      9
NE      8
Name: count, dtype: int64


In [6]:
df.rename(columns={
    'Demo_household_income_bracket_Symbol': 'Demo_income_Symbol',
    'Demo_past_vote_vote_choice_in_the_2020_presidential_election_Symbol': 'Demo_past_vote_Symbol',
    'Demo_highest_educational_qualification_Symbol': 'Demo_education_Symbol'
}, inplace=True)


In [10]:
import pickle

def encode_var(df, col):
    le = LabelEncoder()
    notna_mask = df[col].notna()
    encoded = pd.Series(np.nan, index=df.index)
    encoded[notna_mask] = le.fit_transform(df.loc[notna_mask, col]) + 1

    mapping = {
        original: int(code + 1)
        for code, original in enumerate(le.classes_)
    }

    return encoded, mapping

predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_education_Symbol",
    "Demo_past_vote_Symbol"
]

all_mappings = {}

for col in predictor_cols:
    encoded, mapping = encode_var(df, col)
    new_col_name = col.replace('_Symbol', '') + '_idx'
    df[new_col_name] = encoded
    all_mappings[col] = mapping

print(all_mappings)


shp = gpd.read_file("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/tl_2025_us_state/tl_2025_us_state.shp")
shp["STATE"] = shp["STUSPS"].str.upper()

shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])]

shp = shp.sort_values("STATE").reset_index(drop=True)

state_order = shp["STATE"].tolist()

state_to_idx = {s: i+1 for i, s in enumerate(state_order)}

df["state_cleaned_idx"] = (
    df["state_cleaned"]
      .str.upper()
      .map(state_to_idx)
)


with open('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/cleaned_MrP_data/all_mappings.pkl', 'wb') as f:
    pickle.dump(all_mappings, f)

{'Demo_sex_Symbol': {'s1': 1, 's2': 2}, 'Demo_age_Symbol': {'a1': 1, 'a2': 2, 'a3': 3, 'a4': 4, 'a5': 5, 'a6': 6, 'a7': 7}, 'Demo_ethnicity_Symbol': {'e1': 1, 'e2': 2, 'e3': 3, 'e4': 4, 'e5': 5, 'e6': 6}, 'Demo_income_Symbol': {'h1': 1, 'h2': 2, 'h3': 3, 'h4': 4, 'h5': 5}, 'Demo_education_Symbol': {'q1': 1, 'q2': 2}, 'Demo_past_vote_Symbol': {'vpa1': 1, 'vpa2': 2, 'vpa3': 3, 'vpa4': 4, 'vpa5': 5, 'vpa6': 6}}


In [11]:
df = df.loc[df['Demo_age_idx'] != 1].copy() # removing samples under 18
df.loc[:, 'Demo_age_idx'] = df['Demo_age_idx'] - 1 # shift all age idx down by 1 (starting from 18-24 instead of under 18)

In [12]:
print(df['Demo_age_idx'].unique())

[nan  1.  3.  4.  5.  2.  6.]


In [13]:
df['Demo_age_idx'].dtype
df['Demo_age_idx'].value_counts(dropna=False)


Demo_age_idx
NaN    13498
3.0     1909
2.0     1773
5.0     1309
4.0     1219
6.0      767
1.0      403
Name: count, dtype: int64

In [16]:
df.to_csv('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/drop_usa_cleaned_all_result.csv')

In [18]:
df.to_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [19]:
print(len(df))

20878
